In [1]:
!python -m pip install pyaudio


Defaulting to user installation because normal site-packages is not writeable


In [2]:
!python -m pip install SpeechRecognition

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import cv2
import speech_recognition as sr
import threading

# -----------------------------
# Speech Recognition Setup
# -----------------------------
recognizer = sr.Recognizer()
mic = sr.Microphone()

transcript = ""
listening = True

def speech_to_text():
    global transcript, listening

    with mic as source:
        recognizer.adjust_for_ambient_noise(source)

    while listening:
        try:
            with mic as source:
                audio = recognizer.listen(source, phrase_time_limit=5)

            text = recognizer.recognize_google(audio)
            transcript = text

        except sr.UnknownValueError:
            transcript = "Could not understand..."
        except sr.RequestError:
            transcript = "API error"

# Run speech thread
t = threading.Thread(target=speech_to_text)
t.daemon = True
t.start()

# -----------------------------
# Face Detection (OpenCV)
# -----------------------------
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        cv2.putText(frame,
                    "User Detected",
                    (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2)

    # -----------------------------
    # Display Speech Text
    # -----------------------------
    cv2.putText(frame,
                "Speech Transcription:",
                (20, 400),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 255),
                2)

    cv2.putText(frame,
                transcript[:60],
                (20, 430),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2)

    cv2.imshow("Camera + Speech Transcription", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        listening = False
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import speech_recognition as sr

for index, name in enumerate(sr.Microphone.list_microphone_names()):
    print(index, name)